In [1]:
import numpy as np
from numba import njit

from gtsimulation.Global import Units, Constants, codes
from gtsimulation.MagneticFields import AbsBfield
from gtsimulation.Particle.Generators import Distributions, Spectrums
from gtsimulation.Particle import Flux
from gtsimulation.Algos import BunemanBorisSimulator

In [2]:
class dipole_field(AbsBfield):
    def __init__(self, m0: float) -> None:
        self.m0 = m0
        self.m_vector = np.array([0, 0, 1]) * m0
        self.use_meters = True
        self.use_tesla = True


    def CalcBfield(self, x, y, z):
        return np.array(self.__calc_bfield(x, y, z, self.m_vector))
    
    @staticmethod
    @njit(fastmath=True)
    def __calc_bfield(x, y, z, m_vector):
        r_mag2 = x**2 + y**2 + z**2
        r_mag = np.sqrt(r_mag2)
    
        if r_mag == 0.0:
            return 0.0, 0.0, 0.0
        
        dot_product = m_vector[0]*x + m_vector[1]*y + m_vector[2]*z
    
        r3_inv = 1.0 / (r_mag2 * r_mag)       # 1 / r^3
        r5_inv = r3_inv / r_mag2              # 1 / r^5
    
        factor1 = 3.0 * dot_product * r5_inv  
        factor2 = r3_inv                       
    
        Bx = 1e-7 * (factor1 * x - factor2 * m_vector[0]) * 1e9
        By = 1e-7 * (factor1 * y - factor2 * m_vector[1]) * 1e9
        Bz = 1e-7 * (factor1 * z - factor2 * m_vector[2]) * 1e9
    
        return Bx, By, Bz

    def UpdateState(self, newDate):
        pass

    def to_string(self):
        s = f"""Type: Dipole Magnetic Field
        Version: 1.0"""

        return s

In [3]:
dt = 1e-8
n_steps = 1e3
step = {"UseAdaptiveStep": True, "InitialStep": 1e-5}

In [4]:
m0 = 0.5e31 #амплитуда момента магнитного поля
border = 1000000
R_star = 10000
b_field = dipole_field(m0)

In [5]:

T_el_prime = 100 * Units.MeV
radLosses = [True, {"Photons": True, "MinE": Units.GeV, "MaxE": T_el_prime}]

latitude = 0
r_0 = np.array([np.cos(latitude), 0, np.sin(latitude)]) * R_star 
v_0 = np.array([np.cos(latitude), 0, np.sin(latitude)])


particle = Flux(
    Spectrum=Spectrums.Monolines(energy=T_el_prime),
    Distribution=Distributions.UserInput(
        R0=r_0,
        V0=v_0
    ),
    Names="e-",
    Nevents=1
)

b_field = dipole_field(m0)

simulator = BunemanBorisSimulator(
    Bfield=b_field,
    Medium=None,
    Particles=particle,
    InteractNUC=None,
    UseDecay=False,
    Step=dt,
    Num=n_steps,
    ForwardTrck=1,
    Save=[1, {"Coordinates": True, "Velocities": True, "Energy": True}],
    Output=None,
    Verbose=2,
    # BreakCondition={"Rmax": border},
    RadLosses=radLosses
)

track = simulator()

Creating simulator object...
Date: 2008-01-01 00:00:00
Time step: 1e-08 seconds
Number of steps: 1000
Radiation Losses: True
Synchrotron Emission: True
Region: Undefined
		Additional Energy Losses: False
Number of files: 1
Output file name: None_num.npy
Save every 1 step of:
	Coordinates: True
	Velocities: True
	Efield: False
	Bfield: False
	Angles: False
	Path: False
	Density: False
	Clock: False
	Energy: True
	PitchAngles: False
	LarmorRadii: False
	GuidingCenter: False
Electric field: None
Magnetic field: Type: Dipole Magnetic Field
        Version: 1.0
Medium: None
Decay: False
Nuclear Interactions: None
Flux: 
        Number of particles: 1
        Particles: ['e-']
        V: Isotropic
        Spectrum: Monolines
        Energy: 100
        Distribution: User Input
        R0 shape: (1, 3)
        V0 shape: (1, 3)
Break Conditions:
	Xmin: 0.0
	Ymin: 0.0
	Zmin: 0.0
	Rmin: 0.0
	Dist2Path: 0.0
	Xmax: inf
	Ymax: inf
	Zmax: inf
	Rmax: inf
	MaxPath: inf
	MaxTime: inf
	MaxRev: inf
	Deat

/home/hagib/Templates/gt_env/lib/python3.12/site-packages/gtsimulation/_simulator.py:897: RuntimeWarning: invalid value encountered in divide
  sv[VCode] = Vm / V_norm
/home/hagib/Templates/gt_env/lib/python3.12/site-packages/gtsimulation/Interaction/GenSynchCounter.py:6: RuntimeWarning: invalid value encountered in scalar divide
  cos_BV = np.dot(B, V) / (np.linalg.norm(B) * np.linalg.norm(V))
/home/hagib/Templates/gt_env/lib/python3.12/site-packages/gtsimulation/Interaction/RadLossStep.py:15: RuntimeWarning: invalid value encountered in divide
  Vinter = (Vp + Vm) / Vn


ZeroDivisionError: division by zero

In [6]:
energies_e = track[0][0]["Track"]["Energy"]
photons = track[0][0]["Child"]
energies_p = list(map(lambda p: p["Particle"]['T'], photons))
np.mean(energies_p)

/home/hagib/Templates/gt_env/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/hagib/Templates/gt_env/lib/python3.12/site-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


np.float64(nan)

In [8]:
print(track[0][0])

{'Track': {'Coordinates': array([[ 1.00000000e+04,  0.00000000e+00,  0.00000000e+00],
       [ 1.00012549e+04, -2.72259532e+00,  0.00000000e+00],
       [ 1.00007744e+04, -5.68172942e+00,  0.00000000e+00],
       [ 1.00003106e+04, -8.64351676e+00,  0.00000000e+00],
       [ 1.00010843e+04, -1.15398453e+01,  0.00000000e+00],
       [ 9.99847855e+03, -1.30221916e+01,  0.00000000e+00],
       [ 9.99789583e+03, -1.00814863e+01,  0.00000000e+00],
       [ 1.00002012e+04, -1.19978651e+01,  0.00000000e+00],
       [ 1.00026955e+04, -1.03347501e+01,  0.00000000e+00],
       [ 1.00035026e+04, -1.32219431e+01,  0.00000000e+00]]), 'Velocities': array([[ 1.        ,  0.        ,  0.        ],
       [ 0.41859768, -0.90817178,  0.        ],
       [-0.16026734, -0.98707365,  0.        ],
       [-0.15471794, -0.98795868,  0.        ],
       [ 0.2580795 , -0.96612368,  0.        ],
       [-0.86919816, -0.49446391,  0.        ],
       [-0.19437964,  0.98092638,  0.        ],
       [ 0.76900447, -

In [5]:
from PhotonSimulator import PhotonSimulator

T = 100 * Units.MeV
latitude = 0
r_0 = np.array([10000, 0, 0]) 
v_0 = np.array([np.cos(latitude), 0, np.sin(latitude)])

ph = Flux(
    Spectrum=Spectrums.Monolines(energy=T),
    Distribution=Distributions.UserInput(
        R0=r_0,
        V0=v_0
    ),
    PDGcode=22,
    Nevents=1
)

b_field = dipole_field(m0)

sim = PhotonSimulator(
    Bfield=b_field,
    Particles=ph,
    Step=dt,
    Num=n_steps,
    Save=[1, {"Coordinates": True, "Velocities": True, "Energy": True}],
    Verbose=2,
    PMConversion=True,
)

Creating simulator object...
Date: 2008-01-01 00:00:00
Time step: 1e-09 seconds
Number of steps: 1000
Radiation Losses: False
Synchrotron Emission: False
Region: Undefined
		Additional Energy Losses: False
Number of files: 1
Output file name: None_num.npy
Save every 1 step of:
	Coordinates: True
	Velocities: True
	Efield: False
	Bfield: False
	Angles: False
	Path: False
	Density: False
	Clock: False
	Energy: True
	PitchAngles: False
	LarmorRadii: False
	GuidingCenter: False
Electric field: None
Magnetic field: Type: Dipole Magnetic Field
        Version: 1.0
Medium: None
Decay: False
Nuclear Interactions: None
Flux: 
        Number of particles: 1
        V: Isotropic
        Spectrum: Monolines
        Energy: 1000000
        Distribution: User Input
        R0 shape: (1, 3)
        V0 shape: (1, 3)
Tracing: Inward
Break Conditions:
	Xmin: 0.0
	Ymin: 0.0
	Zmin: 0.0
	Rmin: 0.0
	Dist2Path: 0.0
	Xmax: inf
	Ymax: inf
	Zmax: inf
	Rmax: inf
	MaxPath: inf
	MaxTime: inf
	MaxRev: inf
	Death: i

In [6]:
track = sim()[0][0]

Launching simulation...

File 1/1 started

Event 1/1 started
Particle: gamma (M = 0.000000 [MeV/c2], Z = 0)
Energy: 1000000.000000 [MeV], Rigidity: inf [GV]
Coordinates: [10000.     0.     0.] [m]
Velocity: [1. 0. 0.]
Beta: 1.0
Beta * dt: 29.979246 [m]
Calculating:
Pair Production
Epsilon : 0.8762945606933106
### Break due to Death ###
	Progress: 100%
Event 1/1 finished in 0.833 seconds

Simulation completed!


In [7]:
r = track["Track"]["Coordinates"]
ch = track["Child"]
    
print(ch)

[{'Track': {'Coordinates': array([10029.9792458,     0.       ,     0.       ]), 'Velocities': array([1., 0., 0.])}, 'Particle': {'PDG': 11, 'M': 0.5109989461, 'T': np.float64(876294.0496943644), 'Gen': 1}}, {'Track': {'Coordinates': array([10029.9792458,     0.       ,     0.       ]), 'Velocities': array([1., 0., 0.])}, 'Particle': {'PDG': -11, 'M': 0.5109989461, 'T': np.float64(123704.92830774332), 'Gen': 1}}]
